# Premiers pas avec Spark sur Databricks

Ce notebook parcourt les opérations de base de PySpark sur le jeu de données d'exemple **NYC Taxi** (`samples.nyctaxi.trips`) :

1. Connexion au cluster serverless
2. Lecture et exploration d'une table
3. Filtrage, sélection et tri
4. Création de nouvelles colonnes
5. Agrégation par jour
6. Sauvegarde du résultat dans une table

## 1. Connexion à Databricks

On crée une `SparkSession` via **Databricks Connect** : le code s'exécute localement dans VS Code, mais les calculs sont envoyés au compute **serverless** du workspace, en utilisant le profil `omar-pat` défini dans `~/.databrickscfg`.

On importe aussi `pyspark.sql.functions` sous l'alias `F`, qui contient toutes les fonctions de colonnes (`col`, `round`, `sum`, `avg`, ...).

In [ ]:
try:
    spark  # sur Databricks, spark existe déjà
except NameError:
    # dans VS Code, on se connecte via Databricks Connect
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.profile("omar-pat").serverless().getOrCreate()

from pyspark.sql import functions as F

## 2. Lecture et exploration des données

- `spark.read.table(...)` crée un **DataFrame** qui pointe vers la table. Rien n'est encore lu : Spark est *paresseux* (lazy) et n'exécute les calculs que lorsqu'une **action** est demandée.
- `printSchema()` affiche les colonnes et leurs types.
- `count()` est une action : elle déclenche un vrai calcul sur le cluster pour compter les lignes.
- `limit(5).toPandas()` ramène 5 lignes en local sous forme de DataFrame pandas, pour un affichage lisible dans le notebook.

In [ ]:
trajets = spark.read.table("samples.nyctaxi.trips")

trajets.printSchema()
print("Nombre total de trajets :", trajets.count())
trajets.limit(5).toPandas()

## 3. Filtrer, sélectionner et trier

On cherche les **longs trajets bon marché** : plus de 10 miles pour moins de 20 $.

- `filter(...)` garde les lignes qui respectent la condition. Les conditions se combinent avec `&` (ET) et `|` (OU), chacune entre parenthèses.
- `select(...)` ne conserve que les colonnes utiles.
- `orderBy(... .desc())` trie par distance décroissante.

Ce sont uniquement des **transformations** : Spark construit un plan d'exécution, mais aucune donnée n'est encore calculée.

In [ ]:
longs_trajets_pas_chers = (
    trajets
    .filter((F.col("trip_distance") > 10) & (F.col("fare_amount") < 20))
    .select("tpep_pickup_datetime", "trip_distance", "fare_amount")
    .orderBy(F.col("trip_distance").desc())
)

C'est cette **action** (`toPandas()`) qui déclenche réellement l'exécution du plan défini ci-dessus et affiche les 10 premiers résultats.

In [ ]:
longs_trajets_pas_chers.limit(10).toPandas()


## 4. Enrichir les données avec de nouvelles colonnes

`withColumn(nom, expression)` ajoute (ou remplace) une colonne :

- **`prix_par_mile`** : prix de la course divisé par la distance, arrondi à 2 décimales. `try_divide` renvoie `null` au lieu d'une erreur quand la distance vaut 0.
- **`date`** : on extrait le jour (sans l'heure) de l'horodatage de prise en charge avec `to_date`, pour pouvoir regrouper par jour ensuite.

In [ ]:
trajets_enrichis = (
    trajets
    .withColumn(
        "prix_par_mile",
        F.round(F.try_divide(F.col("fare_amount"), F.col("trip_distance")), 2),
    )
    .withColumn("date", F.to_date("tpep_pickup_datetime"))
)

trajets_enrichis.select("date", "trip_distance", "fare_amount", "prix_par_mile").limit(5).toPandas()

## 5. Agréger par jour

`groupBy("date")` regroupe les trajets par jour, puis `agg(...)` calcule plusieurs indicateurs pour chaque groupe :

| Colonne | Calcul |
|---|---|
| `nb_trajets` | nombre de trajets (`count("*")`) |
| `chiffre_affaires` | somme des prix des courses |
| `distance_moyenne` | distance moyenne parcourue |
| `prix_max` | course la plus chère de la journée |

`alias(...)` donne un nom lisible à chaque colonne calculée, et `orderBy("date")` trie le résultat chronologiquement.

In [ ]:
# ===== Cellule 5 : agréger par jour =====
par_jour = (
    trajets_enrichis
    .groupBy("date")
    .agg(
        F.count("*").alias("nb_trajets"),
        F.round(F.sum("fare_amount"), 2).alias("chiffre_affaires"),
        F.round(F.avg("trip_distance"), 2).alias("distance_moyenne"),
        F.max("fare_amount").alias("prix_max"),
    )
    .orderBy("date")
)

par_jour.limit(10).toPandas()

## 6. Sauvegarder le résultat dans une table

`saveAsTable` écrit le DataFrame agrégé comme table **Delta** dans Unity Catalog, sous `workspace.default.taxi_par_jour` (format `catalogue.schéma.table`).

Le mode `"overwrite"` remplace la table si elle existe déjà : on peut donc relancer le notebook sans erreur. La table est ensuite consultable depuis le Catalog Explorer ou en SQL.

In [ ]:
par_jour.write.mode("overwrite").saveAsTable("workspace.default.taxi_par_jour")